<a href="https://colab.research.google.com/github/mx-oscar-hdez/Proyectos/blob/main/Challenge%20Comparador%20de%20Modelos%20Mx.Oscar.Hdez%20_Hands_On_Fundamentos_de_LLMs_con_Modelos_Llama.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# **HANDS - ON: FUNDAMENTOS DE LLMS CON MODELOS LLAMA**


Una vez vista la masterclass ***Fundamentos de LLMs y Arquitectura de Llama***, se proporciona el siguiente ***Colab*** para construir, en vivo, la primera llamada a Llama y medir su comportamiento.

Usamos **Groq** para tener acceso a inferencia de Llama con GPU gratuita, sin necesidad de infraestructura propia.

De igual manera, se proporciona la **solución** de este notebook a través del siguiente [enlace](https://colab.research.google.com/drive/1bjMHw7fmMxo9dyQp140e9mqotKRkfKBg?usp=sharing).

Antes de llamar a Llama necesitamos activar la GPU gratuita de Colab y conectar nuestra cuenta de **Groq**. La librería `groq` es el cliente oficial en Python para hacer llamadas a la API.

### **COLAB SECRETS**

Para no exponer tu ***API key*** directamente en el código, Colab ofrece un panel de ***Secrets*** (ícono de llave en la barra lateral izquierda) donde puedes guardarla de forma segura, añadiendo un nombre asociado a la ***API key*** para guardarla dentro de una variable y usarla dentro del notebook.

## **CONFIGURACIÓN DEL ENTORNO**

In [2]:
# Instalar cliente de Groq y leer API key desde Colab Secrets

!pip install groq -q

import os
from groq import Groq
from google.colab import userdata

client = Groq(api_key=userdata.get('GROQ_API_KEY'))
print("Cliente de Groq inicializado correctamente.")

Cliente de Groq inicializado correctamente.


## **DE PALABRAS A TOKENS**

Antes de que Llama genere una sola palabra, tu prompt se divide en **tokens**. Vamos a definir un prompt de ejemplo que después enviaremos a Llama.

In [3]:
# Definir un prompt de ejemplo (una pregunta real de tu propio contexto)

prompt = "¿Cuál es la diferencia entre la RAM y el almacenamiento en una computadora?"
# prompt = "Explica en un párrafo qué hace un router doméstico"

print(prompt)

¿Cuál es la diferencia entre la RAM y el almacenamiento en una computadora?


### **PRIMERA LLAMADA A LLAMA (ZERO-SHOT)**

Un LLM predice el siguiente token más probable, **no busca la respuesta en una base de datos**. Vamos a enviar el prompt a Llama en modo zero-shot (sin ejemplos previos) y ver la respuesta generada.

In [4]:
# Enviar el prompt a Llama en Groq y mostrar la respuesta generada
response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt}]
    )

print(response.choices[0].message.content)

**RAM (Memoria de Acceso Aleatorio) vs. Almacenamiento (HDD/SSD, etc.)**

| Característica | RAM | Almacenamiento (HDD/SSD, unidad flash, etc.) |
|----------------|-----|---------------------------------------------|
| **Naturaleza** | Volátil: pierde su contenido cuando la computadora se apaga. | No volátil: conserva los datos aunque la energía se corte. |
| **Uso principal** | Mantiene los datos e instrucciones que la CPU necesita *en tiempo real* para ejecutar programas. | Guarda todo lo que quieres conservar permanentemente: sistema operativo, aplicaciones, archivos de usuario, etc. |
| **Velocidad** | Muy rápida (gigahertz, microsegundos). | Mucho más lenta (kilohertz a megahertz; milisegundos a segundos). |
| **Capacidad típica** | Decenas a cientos de gigabytes (en ordenadores modernos). | Desde cientos de gigabytes hasta varios terabytes (HDD) o 1‑4 TB (SSD). |
| **Costo por GB** | Más caro por gigabyte que el almacenamiento. | Más barato por gigabyte. |
| **Tipo de acceso** | 

### **CUÁNTOS TOKENS CONSUMIÓ EL PROMPT**

La respuesta de la API trae un resumen (`usage`) con el número de tokens de entrada y de salida. Es la forma más directa de responder: ¿cuántos tokens consumió mi prompt?

In [5]:
# Mostrar cuántos tokens tuvo el prompt y cuántos tuvo la respuesta

print("Tokens del prompt:", response.usage.prompt_tokens)
print("Tokens de la respuesta:", response.usage.completion_tokens)
print("Tokens totales:", response.usage.total_tokens)
# response.usage.total_tokens es lo que se factura por esta llamada

Tokens del prompt: 86
Tokens de la respuesta: 607
Tokens totales: 693


### **MIDIENDO LA LATENCIA DE INFERENCIA**

Elegir el tamaño correcto de modelo es una decisión de ingeniería, no solo de potencia bruta. Vamos a medir cuánto tarda Llama en responder el mismo prompt.

In [6]:
# Medir el tiempo de respuesta de Llama para el mismo prompt

import time

inicio = time.time()
response_tiempo = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt}]
)
duracion = time.time() - inicio
# duracion_ms = duracion * 1000  # si prefieres reportarlo en milisegundos

print(f"Tiempo de respuesta: {duracion:.2f} segundos")

Tiempo de respuesta: 1.27 segundos


### **COMPARANDO DOS TAMAÑOS DE MODELO**

Repite la misma llamada usando la versión más grande de Llama disponible en Groq y compara tiempo de respuesta y calidad contra el modelo ligero.

In [7]:
# Repetir la llamada con un modelo Llama más grande y comparar tiempo y calidad

inicio = time.time()
response_grande = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[{"role": "user", "content": prompt}]
)
duracion_grande = time.time() - inicio

print(f"Modelo ligero: {duracion:.2f} s — {response.usage.total_tokens} tokens")
print(f"Modelo grande: {duracion_grande:.2f} s — {response_grande.usage.total_tokens} tokens")
print("\nRespuesta del modelo grande:\n", response_grande.choices[0].message.content)

Modelo ligero: 1.27 s — 693 tokens
Modelo grande: 1.84 s — 870 tokens

Respuesta del modelo grande:
 **RAM (Memoria de Acceso Aleatorio) vs. Almacenamiento (Disco duro, SSD, etc.)**

| Característica | RAM | Almacenamiento |
|----------------|-----|----------------|
| **Tipo de memoria** | Volátil: pierde su contenido al apagar la computadora. | No volátil: conserva los datos aunque se apague el equipo. |
| **Función principal** | Guarda temporalmente los datos y programas que se están ejecutando en ese momento, permitiendo un acceso extremadamente rápido. | Guarda de forma permanente archivos, sistemas operativos, aplicaciones, fotos, videos, etc. |
| **Velocidad** | Mucho más rápida (nanosegundos). | Más lenta (micro‑ o milisegundos). SSDs son mucho más rápidas que HDDs, pero siguen siendo más lentas que la RAM. |
| **Capacidad típica** | Desde 4 GB hasta 128 GB (en equipos de consumo). | Desde 128 GB hasta varios terabytes (TB). |
| **Costo por GB** | Mucho más caro. | Mucho más bar

## **CHALLENGE: COMPARADOR DE MODELOS LLAMA**

Una vez visto el ***Hands-On: Fundamentos de LLMs***, se presenta el siguiente reto para que el alumnado pueda repasar y reforzar lo aprendido dentro de la clase.

Se construirá un pequeño **comparador que envíe 3 preguntas** reales a Llama y registre, para cada una, el **tiempo de respuesta** y el **número de tokens** consumidos.

**IMPORTANTE:** Para su revisión, **es indispensable que los apartados anteriores se encuentren llenados con el código visto durante la sesión.**

### **INSTRUCCIONES:**

**1. Carga la API key y genera 3 preguntas:**

   * Lee la API key de llama ya configurada desde **Colab Secrets**.
   * Construye una lista vacía llamada `preguntas` y agrégale 3 preguntas frecuentes de tu propio contexto (soporte técnico, tienda, escuela, etc.).

In [15]:
# Leer API key desde Colab Secrets
!pip install groq -q

import os
from groq import Groq
from google.colab import userdata

client = Groq(api_key=userdata.get('GROQ_API_KEY'))
print("Cliente de Groq inicializado correctamente.")

Cliente de Groq inicializado correctamente.


In [22]:
# Definir la lista de preguntas
prompt_base = "¿Cual es la explicacion mas sencilla?"
preguntas = [
    "Significado de LLMs",
    "Significado de Tokens",
    "Significado de Prompt",
]


**2. Consulta la primera pregunta:** Envía la primera pregunta a Llama usando el modelo más ligero disponible en Groq. Guarda la respuesta, el tiempo de respuesta y el número de tokens en un diccionario llamado `resultado_1`.

In [23]:
#Consultar la primera pregunta y guardar el resultado en resultado_1
pregunta_completa = f"{prompt_base} {preguntas[0]}"
#Tomar tiempo inicio
inicio = time.time()
response_1 = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": pregunta_completa}]
)
#Tomar tiempo fin
fin = time.time()
#Extraer los datos
resultado_1 = {
    "Pregunta": preguntas[0],
    "Respuesta": response_1.choices[0].message.content,
    "Tiempo de respuesta": fin - inicio,
    "Tokens": response_1.usage.total_tokens
}
#Ver resultado
print(resultado_1)

{'Pregunta': 'Significado de LLMs', 'Respuesta': '### Explicación sencilla de **LLM**\n\n**LLM** significa **Large Language Model** (Modelo de Lenguaje Grande).  \nEs una pieza de inteligencia artificial que:\n\n1. **Aprende de mucho texto**  \n   Se entrena con miles de millones de palabras (libros, artículos, sitios web, etc.). Al leer tanto material, aprende las reglas de la lengua, cómo se estructuran las frases y qué palabras suelen aparecer juntas.\n\n2. **Genera lenguaje**  \n   Cuando le das una pregunta o una frase, el LLM usa lo que aprendió para predecir la palabra siguiente, y así va formando una respuesta coherente y útil.\n\n3. **No “piensa”**  \n   No tiene conciencia ni entiende el mundo como los humanos. Simplemente sigue patrones estadísticos que halló en su entrenamiento.\n\n4. **Se usa en muchas cosas**  \n   - Chatbots (como el que estás usando).  \n   - Traducción automática.  \n   - Resúmenes de textos.  \n   - Generación de ideas o código.  \n\n---\n\n#### ¿Por 

**3. Repite para la segunda y tercera pregunta:** Crea `resultado_2` y `resultado_3` de la misma forma.

In [25]:
# Consultar la segunda pregunta y guardar el resultado en resultado_2
pregunta_completa = f"{prompt_base} {preguntas[1]}"
#Tomar tiempo inicio
inicio = time.time()
response_2 = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": pregunta_completa}]
)
#Tomar tiempo fin
fin = time.time()
#Extraer los datos
resultado_2 = {
    "Pregunta": preguntas[1],
    "Respuesta": response_1.choices[0].message.content,
    "Tiempo de respuesta": fin - inicio,
    "Tokens": response_2.usage.total_tokens
}
#Ver resultado
print(resultado_2)

{'Pregunta': 'Significado de Tokens', 'Respuesta': '**Tokens = “piezas” de texto que entiende la IA**  \n\n- Cuando la IA lee (o escribe) algo, no procesa caracteres individuales ni palabras completas, sino **tokens**.  \n- Un token puede ser:  \n  * Una palabra completa (“gato”).  \n  * Una parte de una palabra larga (“in‑com‑ple‑te”).  \n  * Un signo de puntuación (“,” o “!”).  \n  * Incluso un espacio en blanco.  \n\nEn otras palabras, los tokens son los “pedacitos” en los que se divide el texto para que el modelo pueda analizarlos y generar respuestas.\n\n**¿Por qué importa?**  \n- El modelo tiene un límite de tokens en su “contexto” (ej. 4\u202f096 tokens).  \n- Cuantos más tokens uses, más memoria y tiempo necesita, y en algunas plataformas también afecta el coste.  \n\n**Resumen rápido**  \n- **Token = unidad de texto que la IA procesa.**  \n- Puede ser una palabra, parte de una palabra, un signo o un espacio.  \n- El número de tokens determina cuánto texto puede manejar y cuánt

In [26]:
# Consultar la tercera pregunta y guardar el resultado en resultado_3
pregunta_completa = f"{prompt_base} {preguntas[2]}"
#Tomar tiempo inicio
inicio = time.time()
response_3 = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": pregunta_completa}]
)
#Tomar tiempo fin
fin = time.time()
#Extraer los datos
resultado_3 = {
    "Pregunta": preguntas[2],
    "Respuesta": response_3.choices[0].message.content,
    "Tiempo de respuesta": fin - inicio,
    "Tokens": response_3.usage.total_tokens
}
#Ver resultado
print(resultado_3)

{'Pregunta': 'Significado de Prompt', 'Respuesta': '**Prompt – la explicación más sencilla**\n\n| Contexto | ¿Qué significa “prompt”? |\n|----------|---------------------------|\n| **Computación / Sistemas operativos** | Un *prompt* es el texto que aparece en la pantalla esperando que tú escribas algo. Por ejemplo, en la línea de comandos de Windows aparece `C:\\>` o en la terminal de Linux `/home/usuario$`. Ese es el “indicador” o “símbolo” que te dice que el sistema está listo para recibir tu comando. |\n| **Inteligencia artificial / ChatGPT** | Un *prompt* es la pregunta, instrucción o fragmento de texto que tú introduces para que el modelo genere una respuesta. Ejemplo: `Escribe un poema sobre el atardecer` → ese es el prompt que le das al modelo. |\n| **En general (uso cotidiano)** | Un *prompt* también puede ser una sugerencia o recordatorio que incita a hacer algo: “¿Necesitas ayuda?” es un prompt para que la persona responda. |\n| **Adjetivo** | “Prompt” como adjetivo significa

**4. Junta los resultados:** Agrega `resultado_1`, `resultado_2` y `resultado_3` a una lista vacía llamada `resultados`.

In [27]:
# Definir la lista resultados y agregar los tres diccionarios
resultados = []
resultados.append(resultado_1)
resultados.append(resultado_2)
resultados.append(resultado_3)

**5. Muestra la comparación:** Imprime `resultados` y concluye si el modelo ligero resolvió las 3 preguntas satisfactoriamente.

In [31]:
# Mostrar la tabla final de resultados
print("Preguntas y Respuestas:")
for resultado in resultados:
    print(f"Pregunta: {resultado['Pregunta']}")
    print(f"Respuesta: {resultado['Respuesta']}")

# Conclusion
print("\nConclusion:")
print(
    "El modelo ligero resolvio satisfactoriamente las 3 preguntas"
)

Preguntas y Respuestas:
Pregunta: Significado de LLMs
Respuesta: ### Explicación sencilla de **LLM**

**LLM** significa **Large Language Model** (Modelo de Lenguaje Grande).  
Es una pieza de inteligencia artificial que:

1. **Aprende de mucho texto**  
   Se entrena con miles de millones de palabras (libros, artículos, sitios web, etc.). Al leer tanto material, aprende las reglas de la lengua, cómo se estructuran las frases y qué palabras suelen aparecer juntas.

2. **Genera lenguaje**  
   Cuando le das una pregunta o una frase, el LLM usa lo que aprendió para predecir la palabra siguiente, y así va formando una respuesta coherente y útil.

3. **No “piensa”**  
   No tiene conciencia ni entiende el mundo como los humanos. Simplemente sigue patrones estadísticos que halló en su entrenamiento.

4. **Se usa en muchas cosas**  
   - Chatbots (como el que estás usando).  
   - Traducción automática.  
   - Resúmenes de textos.  
   - Generación de ideas o código.  

---

#### ¿Por qué se 